In [3]:

import os
import scipy.io
import h5py
import numpy as np
import sys
from scipy import signal as sig
from scipy import stats as stats
import math
import pandas as pd
import csv
import matplotlib.pyplot as plt
from scipy import signal
import seaborn as sns
import logging

%matplotlib widget

scripts_path = '/home/apollo/Documents/github_projects/ripple_sync/scripts/hc_18_scripts/'
data_path = "/media/apollo/projects/hc-18/data/"
base_path = "/media/apollo/projects/hc-18/data/"

os.chdir(scripts_path)

import hc18_BaseFunctions as hc18_bf
import ripple_detection as ripple
import normalizing_functions as norm
import helper_functions as helper
import bootstrapped_estimation as bootstrap


sessions = ['Train-242-20140124','Train-261-20140617','Train-272-20141215','Train-292-20150501','Train-314-20160118']
animals = ['242','261','272','292','314']


In [48]:
session = sessions[0]
events = hc18_bf.load_evt(session)
# print(events{'description'})

In [49]:
events['time']


[0.0,
 4808.04082491,
 4819.7114759099995,
 4905.54266527,
 4905.54266527,
 5062.62459456,
 5062.62459456,
 5366.32890743,
 5366.32890743,
 5522.05817366,
 5522.05817366,
 9422.27334576,
 9422.27334576,
 9589.862006680001,
 9589.862006680001,
 9772.88046359,
 9772.88046359,
 13409.98694307,
 13409.98694307,
 13622.70107261,
 13622.70107261,
 13703.215981590001,
 13703.215981590001,
 13913.90111655,
 13913.90111655,
 16089.675367310001,
 16089.675367310001,
 17696.27731656]

In [50]:
events['description']


['beginning of Train-242-20140124-01-SleepBaseline',
 'end of Train-242-20140124-01-SleepBaseline',
 'beginning of Train-242-20140124-03-PassiveForward',
 'end of Train-242-20140124-03-PassiveForward',
 'beginning of Train-242-20140124-04-PassiveForward',
 'end of Train-242-20140124-04-PassiveForward',
 'beginning of Train-242-20140124-06-PassiveForward',
 'end of Train-242-20140124-06-PassiveForward',
 'beginning of Train-242-20140124-07-PassiveForward',
 'end of Train-242-20140124-07-PassiveForward',
 'beginning of Train-242-20140124-08-SleepPassiveForward',
 'end of Train-242-20140124-08-SleepPassiveForward',
 'beginning of Train-242-20140124-09-PseudoActiveForward',
 'end of Train-242-20140124-09-PseudoActiveForward',
 'beginning of Train-242-20140124-10-PseudoActiveForward',
 'end of Train-242-20140124-10-PseudoActiveForward',
 'beginning of Train-242-20140124-11-SleepPseudoActiveForward',
 'end of Train-242-20140124-11-SleepPseudoActiveForward',
 'beginning of Train-242-20140124-

In [6]:

event_name = 'SleepBaseline'
event_name = 'SleepPassiveForward'

for animal_idx,session in enumerate(sessions):

    print(session)
    
    event_times, event_descriptions = hc18_bf.get_events(session, event_name)
    print(event_times)
    print(event_descriptions)
    print()

Train-242-20140124
[5522.05817366, 9422.27334576, 13913.90111655, 16089.675367310001, 16089.675367310001, 17696.27731656]
['beginning of Train-242-20140124-08-SleepPassiveForward', 'end of Train-242-20140124-08-SleepPassiveForward', 'beginning of Train-242-20140124-15-SleepPassiveForward', 'end of Train-242-20140124-15-SleepPassiveForward', 'beginning of Train-242-20140124-16-SleepPassiveForward', 'end of Train-242-20140124-16-SleepPassiveForward']

Train-261-20140617
[5650.466791950001, 8387.54902414, 8387.54902414, 11470.15802339, 16092.05039201, 19964.362956459998]
['beginning of Train-261-20140617-05-SleepPassiveForward', 'end of Train-261-20140617-05-SleepPassiveForward', 'beginning of Train-261-20140617-06-SleepPassiveForward', 'end of Train-261-20140617-06-SleepPassiveForward', 'beginning of Train-261-20140617-12-SleepPassiveForward', 'end of Train-261-20140617-12-SleepPassiveForward']

Train-272-20141215
[4839.59247707, 8666.685201060001, 13603.19755881, 18246.68535036]
['begin

In [13]:

states = ['SleepBaseline','SleepPassiveForward']

output_dict = dict()
output_dict['rat'] = []
output_dict['session'] = []
output_dict['state'] = []
output_dict['shank'] = []
output_dict['ripple_length'] = []
output_dict['ripple_counts'] = []
output_dict['ripple_max_amp'] = []

for animal_idx,session in enumerate(sessions):

    print(session)
    
    lfp,srate = hc18_bf.load_lfp_hc18_data(session)

    for state in states:
        event_times, event_descriptions = hc18_bf.get_events(session, state)
                
        if state == 'SleepPassiveForward' and session == 'Train-292-20150501':
            event_start = event_times[0]
            event_end = event_times[3]
    
        elif event_name == 'SleepPassiveForward':
            event_start = event_times[0]
            event_end = event_times[1]
    
        else:
            event_start = np.nanmin(event_times)
            event_end = np.nanmax(event_times)
            
        print(event_start)
        print(event_end)
        print()
    
        file_path = f'{data_path}/{session}/'
        filename = f'{session}_{state}_sws_epoch'
        sleep_epochs = np.load(file_path + filename)
        lfp = lfp[:,sleep_epochs]
        lfp = lfp[:,0:int(30*60*srate)]
        
        shanks_electrodes,left_shanks,right_shanks = hc18_bf.get_shanks_info(session)
        
        max_ripple_channels = []
        for channels in shanks_electrodes:
            if len(channels) > 0:
                ripple_lfp = lfp[channels,:]
                ripple_filtered = sig.detrend(helper.eegfilt(ripple_lfp,srate,100,250))
                ripple_amp = np.abs(helper.hilbert(ripple_filtered))
        
                max_ripple_idx = np.argmax(np.nanmean(ripple_amp,1))
                max_ripple_channels.append(channels[max_ripple_idx])
        
            else:
                max_ripple_channels.append(np.nan)
        max_ripple_channels = np.array(max_ripple_channels)
        del ripple_filtered, ripple_amp
    
        for shank_1 in np.arange(0,len(shanks_electrodes)):    
            
            print('Shank 1 = ' + str(shank_1))
            ripple_channel_1 = max_ripple_channels[shank_1]
            if np.isnan(ripple_channel_1):
                continue
    
            ripple_lfp_1 = lfp[int(ripple_channel_1),:]
    
            ripple_features = ripple.get_ripple_info(ripple_lfp_1, srate, ripple_band=(100, 250),ripple_threshold_level = 2, 
                                             min_ripple_duration = 0.02, max_ripple_duration = 0.2, smooth_time = 0.005)
                
            ripple_length = []
            for cyc in ripple_features['cycles']:
                ripple_length.append(cyc.shape[0]/srate)
            ripple_length = np.nanmean(ripple_length)
            ripple_counts = len(ripple_features['centers'])/(30*60)
            ripple_max_amp = np.nanmean(ripple_features['max_amplitudes'])
            
            output_dict['rat'].append(animals[animal_idx])
            output_dict['session'].append(session)
            output_dict['state'].append(state)
            output_dict['shank'].append(shank_1)
            output_dict['ripple_length'].append(ripple_length)
            output_dict['ripple_counts'].append(ripple_counts)
            output_dict['ripple_max_amp'].append(ripple_max_amp)

df = pd.DataFrame(output_dict)
df


Train-242-20140124
0.0
4808.04082491

Shank 1 = 0
Shank 1 = 1
Shank 1 = 2
Shank 1 = 3
Shank 1 = 4
Shank 1 = 5
Shank 1 = 6
Shank 1 = 7
Shank 1 = 8
Shank 1 = 9
Shank 1 = 10
Shank 1 = 11
Shank 1 = 12
Shank 1 = 13


KeyboardInterrupt: 

In [9]:

lfp,srate = hc18_bf.load_lfp_hc18_data(session)


In [12]:
lfp

array([[29119, 18308,  9505, ..., 23435, 30983, 32767],
       [-5303, -5071, -4820, ...,   452,   136,  -111],
       [-4658, -4380, -4721, ...,   343,  -373,  -410],
       ...,
       [-6890, -5995, -5833, ..., -2637, -3098, -3032],
       [-3367, -3077, -3447, ..., -2122, -2015, -1564],
       [-1697,  -874, -1079, ..., -1431, -1520, -1138]],
      shape=(64, 3955000), dtype=int16)

In [11]:

events = hc18_bf.load_evt(session)
events

array([ 6115.1, 11512.2, 18214.3, 23442.1])

In [ ]:

def get_events(session,event_name):
    
    data_path = base_path + session
    events = load_evt(session)
    
    event_times = []
    event_descriptions = []
    for i, description in enumerate(events['description']):
        if event_name in description and ("beginning" in description or "end" in description):
            event_times.append(events['time'][i])
            event_descriptions.append(description)
    return event_times, event_descriptions
